### Προετοιμασία δεδομένων

Το πρώτο κελί προετοιμάζει αυτόματα τα εκπαιδευτικά δεδομένα. Εκτέλεσέ το πριν ξεκινήσεις την άσκηση.


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import json

DATA_ROOT = next((p for p in (Path("../../data"), Path("data"), Path("downloads/python/data")) if p.exists()), Path("data"))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
DATA_URLS = {
    "measurement_areas.geojson": "https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp07/measurement_areas.geojson",
    "measurement_routes.geojson": "https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp07/measurement_routes.geojson",
    "measurement_sites.csv": "https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp07/measurement_sites.csv"
}

def valid_cached(path):
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        if path.suffix.lower() == ".gpkg":
            return path.read_bytes()[:16] == b"SQLite format 3\x00"
        if path.suffix.lower() == ".geojson":
            json.loads(path.read_text(encoding="utf-8"))
        return True
    except Exception:
        return False

for filename, url in DATA_URLS.items():
    target = DATA_ROOT / "gp07" / filename
    if valid_cached(target):
        print(f"[OK] Ήδη διαθέσιμο: {target}")
        continue
    target.parent.mkdir(parents=True, exist_ok=True)
    try:
        with urlopen(url, timeout=30) as response:
            payload = response.read()
        if not payload:
            raise RuntimeError("κενή απάντηση")
        target.write_bytes(payload)
        print(f"[OK] Κατέβηκε: {target}")
    except Exception as exc:
        if target.exists():
            target.unlink()
        print(f"[WARN] Δεν ήταν δυνατή η λήψη {url}: {exc}")


# GP07 — Measurements

Υπολόγισε μήκη, εμβαδά και αποστάσεις σε projected CRS. Μην κάνεις μετρήσεις σε μοίρες. Πριν από κάθε αποτέλεσμα έλεγξε geometry type, CRS και units. Correct code + wrong CRS = wrong measurement.

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
root = DATA_ROOT / "gp07"
areas = gpd.read_file(root / 'measurement_areas.geojson')
routes = gpd.read_file(root / 'measurement_routes.geojson')
sites_df = pd.read_csv(root / 'measurement_sites.csv')
sites = gpd.GeoDataFrame(sites_df, geometry=gpd.points_from_xy(sites_df.longitude, sites_df.latitude), crs='EPSG:4326')


In [ ]:
target_crs = 'EPSG:____'
areas_m = areas.to_crs(target_crs)
routes_m = routes.to_crs(target_crs)
sites_m = sites.to_crs(target_crs)


In [ ]:
areas_m['area_m2'] = areas_m.geometry.area
areas_m['area_ha'] = areas_m['area_m2'] / ____
routes_m['length_m'] = routes_m.geometry.length
routes_m['length_km'] = routes_m['length_m'] / ____


In [ ]:
distance_m = sites_m.geometry.iloc[0].distance(sites_m.geometry.iloc[1])
print('distance:', distance_m, 'm')


QA: θετικά μεγέθη, μέτρα στο projected CRS, σωστό πλήθος features και λογικές τάξεις μεγέθους. Πρόσθεσε min/max/mean ή median όπου βοηθούν να βρεις outlier και εξήγησε γιατί η distance είναι γεωμετρική, όχι network distance.

In [ ]:
# assert ...
